# Street Network Analysis

In the previous section, we looked at how a street network can be represented as a graph and explored its key characteristics: the structure of nodes and edges, node degrees, and network connectivity.

Now we move on to actual network analysis tasks. Using the `networkx` library, we will perform several key types of analysis:

- finding shortest paths,
- evaluating node centrality,
- building a distance matrix,
- building service areas.

## 0. Importing Libraries and Preparing Data

### 0.1. Importing Libraries

In [ ]:
import osmnx as ox
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt

from shapely.geometry import Point

# keep all downloaded OSM responses in one cache folder at the root of the repository
ox.settings.cache_folder = "../../cache"

### 0.2. Preparing Data

Load the street network graph from OSM

In [ ]:
# Define the study area
area_name = "Leninsky District, Yekaterinburg"

# Load the street network graph from OpenStreetMap
# network_type="drive" — road network accessible by car
graph = ox.graph_from_place(area_name, network_type="drive")

# Visualise the graph
ox.plot_graph(graph)

Load cafés from OSM for the same area.

OSM stores some cafés as polygons (a building outline) rather than points. The steps below read point coordinates directly, so we keep only the point features — as we did when filtering geometry types in the first module.

In [ ]:
cafes = ox.features_from_place(area_name, tags={"amenity": "cafe"})

# keep point features only
cafes = cafes[cafes.geom_type == "Point"]

print(f"Cafés: {len(cafes)}")

cafes.explore(tiles="cartodbpositron")

## 1. Shortest Path

One of the most common network analysis tasks is finding the shortest path between locations in the urban environment.

In a street network, the shortest path is determined not along a straight line but along graph edges, taking their weights (such as length or travel time) into account.

Let's find the shortest path between two cafes in the study area.

### 1.1. Checking the Coordinate Reference System

Does the CRS of the café layer match the graph's?

In [ ]:
graph.graph["crs"] == cafes.crs

In this case the coordinate reference systems already match. If they differed, the data would need to be reprojected to a common CRS.

### 1.2. Selecting Points

Let's pick two random cafés to route between.
We sample two features from the cafés layer with `.sample()` and extract their geometries — those coordinates are what we need to build the route. The `random_state` argument fixes the draw, so the notebook produces the same route every time it runs.

In [ ]:
route_cafes = cafes.sample(n=2, random_state=42)
cafe_1 = route_cafes.iloc[0].geometry
cafe_2 = route_cafes.iloc[1].geometry

### 1.3. Finding the Nearest Graph Nodes

Because a street network graph is made up of nodes and edges, we need to snap our selected points to the nearest graph nodes before routing. We use the `nearest_nodes` function, which finds the closest node for a given coordinate.

In [ ]:
orig_node = ox.distance.nearest_nodes(graph, X=cafe_1.x, Y=cafe_1.y)
dest_node = ox.distance.nearest_nodes(graph, X=cafe_2.x, Y=cafe_2.y)

### 1.4. Computing the Shortest Route

Now that we have the origin and destination nodes, we can compute the shortest path between them.

We use `shortest_path` from `networkx` with edge `length` as the weight, so the route is minimised by distance.

The total route length is calculated separately with `shortest_path_length`.

> `shortest_path` returns the sequence of nodes along the route,  
> while `shortest_path_length` returns its total length (in metres, since the `length` attribute is stored in metres).

In [ ]:
route_nodes = nx.shortest_path(graph, source=orig_node, target=dest_node, weight="length")
route_length = nx.shortest_path_length(graph, source=orig_node, target=dest_node, weight="length")

print(f"Path: {route_nodes}")
print(f"Distance (metres): {route_length}")

### 1.5. Visualising the Route

Let's visualise the route on the graph using `plot_graph_route` from `osmnx`.

In [ ]:
ox.plot_graph_route(graph, route_nodes, route_linewidth=2, node_size=0, bgcolor="white")

## 2. Centrality

Another important network analysis task is identifying how significant individual nodes are within the network.

Centrality measures help reveal the most important elements of a street network — for example, nodes through which the greatest number of routes pass, or nodes that provide fast access to other parts of the network.

Several types of centrality can be used depending on the task.

### 2.1. Degree Centrality

**Degree Centrality** measures the number of connections (edges) a node has.

In a street network context, a node's degree corresponds to the number of roads meeting at that intersection.

Nodes with high degree centrality are considered more "connected" and often correspond to major intersections or transport hubs.

In `networkx`, this measure is available via `nx.degree_centrality()`.

Let's calculate degree centrality for all nodes in the graph.

In [ ]:
# Calculate degree centrality
degree_centrality = nx.degree_centrality(graph)

# Visualise the result
fig, ax = plt.subplots(figsize=(10, 10))

# the multiplier only scales the markers so the values are visible on the map
node_sizes = [v * 1000 for v in degree_centrality.values()]

ox.plot_graph(graph, node_size=node_sizes, node_color="red", bgcolor="white", ax=ax, show=False)

ax.set_axis_off()
plt.title("Degree Centrality")
plt.show()

### 2.2. Betweenness Centrality

**Betweenness Centrality** reflects how often a node lies on the shortest paths between other nodes in the network.

In other words, it shows which nodes the greatest number of routes pass through.

Nodes with high betweenness centrality act as "bridges" connecting different parts of the network. In a street network, such nodes often correspond to key intersections that carry significant traffic flow.

Betweenness centrality is computed based on shortest paths between all pairs of nodes. In `networkx`, use `nx.betweenness_centrality()`.

In [ ]:
# Calculate betweenness centrality
betweenness_centrality = nx.betweenness_centrality(graph, weight="length")

# Visualise the result
fig, ax = plt.subplots(figsize=(10, 10))

# the multiplier only scales the markers so the values are visible on the map
node_sizes = [v * 1000 for v in betweenness_centrality.values()]

ox.plot_graph(graph, node_size=node_sizes, node_color="blue", bgcolor="white", ax=ax, show=False)

ax.set_axis_off()
plt.title("Betweenness Centrality")
plt.show()

### 2.3. Closeness Centrality

**Closeness Centrality** reflects how close a node is to all other nodes in the network in terms of shortest paths.

In other words, it shows how quickly you can reach other parts of the network from a given node.

Nodes with high closeness centrality have the smallest total distances to all other nodes and provide good accessibility to different parts of the area.

Closeness centrality is calculated based on shortest path lengths from a node to all other nodes. In `networkx`, use `nx.closeness_centrality()`.

Note the larger multiplier for `node_sizes` below: closeness values are much smaller than the other two measures, so they need more scaling to stay visible. The multiplier affects only how the markers are drawn, not the values themselves.

In [ ]:
# Calculate closeness centrality
closeness_centrality = nx.closeness_centrality(graph, distance="length")

# Visualise the result
fig, ax = plt.subplots(figsize=(10, 10))

# the multiplier only scales the markers so the values are visible on the map
node_sizes = [v * 100000 for v in closeness_centrality.values()]

ox.plot_graph(graph, node_size=node_sizes, node_color="green", bgcolor="white", ax=ax, show=False)

ax.set_axis_off()
plt.title("Closeness Centrality")
plt.show()

> **Degree Centrality** reflects the number of connections a node has and helps identify major intersections
>
> **Betweenness Centrality** identifies nodes through which the greatest number of routes pass
>
> **Closeness Centrality** shows how quickly other parts of the network can be reached from a given node

## 3. Distance Matrix

A distance matrix is a data structure that holds the shortest path lengths between a set of points in the network.

It answers the question **how far apart features are along the actual street network**, rather than as the crow flies.

Suppose we need to assess which cafés are closer to one another in terms of distance along the street network.
We'll calculate distances between all pairs of selected points and present the result as a matrix.

### 3.1. Selecting a Subset of Points

For this example, we'll take a few random cafes

In [ ]:
sample_cafes = cafes.sample(n=5, random_state=42)

### 3.2. Snapping to Graph Nodes

For each selected point we find the nearest node in the street network.
This "snaps" the features to the graph so that we can compute distances between them.

In [ ]:
sample_nodes = [
    ox.distance.nearest_nodes(graph, X=geom.x, Y=geom.y)
    for geom in sample_cafes.geometry
]

### 3.3. Computing the Distance Matrix

We'll calculate the distance matrix between the selected graph nodes by finding the shortest paths between all pairs.

The underlying algorithm is **Dijkstra's algorithm** — one of the most widely used shortest-path algorithms in graph theory.
It expands outward from the source node, always selecting the nearest unvisited vertex and updating distances to neighbouring nodes.

`networkx` already implements Dijkstra's algorithm, so we don't need to code it from scratch — we just call the appropriate function.

Here's the approach:

- for each node in the selected set, we run a single-source shortest-path search to all other reachable nodes;
- then we extract the distances only to the nodes of interest;
- finally, we assemble the distance matrix.

In [ ]:
distance_matrix = {}

for i, node_i in enumerate(sample_nodes):
    distances = nx.single_source_dijkstra_path_length(graph, node_i, weight="length")
    
    for j, node_j in enumerate(sample_nodes):
        distance_matrix[(i, j)] = distances.get(node_j, None)

Here:

- `single_source_dijkstra_path_length` computes shortest distances from one node (`node_i`) to all reachable nodes in the graph;
- `weight='length'` means edge length (in metres) is used as the weight;
- in the inner loop we extract distances only to the relevant nodes and store them;
- the result is a structure containing distances between all pairs of points.

We populate the distance matrix step by step, where each index pair corresponds to the distance between two features.

### 3.4. Distance Matrix as a Table

It's convenient to present the distance matrix as a table.

We convert the dictionary of distances into a `pandas` `DataFrame`, where rows and columns correspond to the selected points:

- we create an empty table of size `len(sample_nodes) × len(sample_nodes)`;
- dictionary keys `(i, j)` correspond to row and column indices;
- we write the computed distances into the appropriate cells using `.loc`;
- the result is the distance matrix in tabular form.

In [ ]:
distance_matrix_df = pd.DataFrame(index=range(len(sample_nodes)), columns=range(len(sample_nodes)))

for (i, j), dist in distance_matrix.items():
    distance_matrix_df.loc[i, j] = dist

distance_matrix_df

## 4. Service Areas

A service area is a region reachable from a given point within a specified distance or travel time along the street network.

Unlike buffers, which are drawn using straight-line distance, service areas follow the network structure and give a more realistic picture of how people move through the city.

A special case of service areas is **isochrones** — areas reachable within a given **travel time** (e.g., 5, 10, or 15 minutes).

Let's find out how far you can get from a given point within 1.5 km along the street network. We are using the drivable network here, so this is the area reachable by car; loading the graph with `network_type="walk"` would give the walking equivalent.

### 4.1. Selecting the Starting Point

First, we define the origin point and find its nearest graph node.

In [ ]:
# Sample one random cafe
start_cafe = cafes.sample(n=1, random_state=20).iloc[0].geometry

# Find the nearest graph node to this cafe
start_node = ox.distance.nearest_nodes(graph, X=start_cafe.x, Y=start_cafe.y)

Here:

- `start_cafe` holds the coordinates (longitude and latitude) of the origin point;
- `nearest_nodes` finds the nearest street network node;
- the service area will be computed from this node.

### 4.2. Setting the Radius

Define the maximum distance that determines the service area.

In [ ]:
max_dist = 1500  # e.g., 1.5 km

### 4.3. Building the Service Area (Subgraph)

Now we build a **subgraph** — a portion of the original graph containing only the nodes and edges reachable from the origin within the specified distance.

The `ego_graph` function constructs a subgraph of all nodes reachable from the source node within the given radius:

- the algorithm traverses the network outward from the origin node,
- collects all nodes reachable with a cumulative path length ≤ `max_dist`,
- and returns a new graph (subgraph) containing only those nodes and the edges connecting them.

In [ ]:
subgraph = nx.ego_graph(graph, start_node, radius=max_dist, distance="length")

To better understand how the service area is formed, let's visualise:

- the full street network graph,
- the subgraph — the resulting service area,
- the origin point.

In [ ]:
# Convert the full graph to a GeoDataFrame to get its bounding box
nodes_all, edges_all = ox.graph_to_gdfs(graph)

# Bounding box of the full graph
xmin, ymin, xmax, ymax = edges_all.total_bounds

# Create the figure
fig, ax = plt.subplots(figsize=(10, 10))

# Draw the full graph
ox.plot_graph(
    graph,
    ax=ax,
    node_size=0,
    edge_color="lightgray",
    edge_linewidth=1,
    show=False,
    close=False
)

# Draw the subgraph on top
ox.plot_graph(
    subgraph,
    ax=ax,
    node_size=0,
    edge_color="red",
    edge_linewidth=2,
    show=False,
    close=False
)

# Get coordinates of the start node
start_x = graph.nodes[start_node]["x"]
start_y = graph.nodes[start_node]["y"]

# Mark the start point
ax.scatter(start_x, start_y, c="blue", s=80, zorder=5)

# Fix axis limits to the full graph extent
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.set_axis_off()

plt.show()

### 4.4. Converting the Subgraph to Geometry

The subgraph is still a graph (nodes and edges), not a geometry, so it can't be displayed as a polygon directly. To work with it as a spatial object, we convert it to a GeoDataFrame.

In [ ]:
subgraph_edges = ox.graph_to_gdfs(subgraph, nodes=False, edges=True)

Now we have lines (graph edges) representing the reachable portion of the network.

Merge them into a single geometry:

In [ ]:
service_area = subgraph_edges.geometry.union_all()

At this point we have a set of lines, but a service area is typically represented as a polygon.

> Note: the service area is built from the selected cafe using the street network clipped to the study district boundary. In reality it may extend further, as the network continues beyond those boundaries.

### 4.5. Building the Service Area Polygon

To get a closed area, we wrap the lines in a hull. `convex_hull` creates a convex polygon around the subgraph.

In [ ]:
isochrone = service_area.convex_hull

This is a simplified approximation of the service area — the actual reachable zone may have a more complex shape.

### 4.6. Visualisation

To display the service area and the origin point on an interactive map, we create a GeoDataFrame

In [ ]:
iso_gdf = gpd.GeoDataFrame(geometry=[isochrone], crs=subgraph_edges.crs)
point_gdf = gpd.GeoDataFrame(geometry=[Point(start_x, start_y)], crs=subgraph_edges.crs)

View on the map

In [ ]:
m = iso_gdf.explore(color="red", alpha=0.5, tiles="cartodbpositron")
point_gdf.explore(m=m, color="blue", markersize=100)


## Summary

In this section we covered the core tasks of street network analysis.

We learned how to:

- find **shortest paths** between locations;
- assess **node importance** using centrality measures;
- compute a **distance matrix** between features;
- build **service areas** and analyse territorial coverage.

All the methods covered here rely on an explicit graph representation of the street network, which we obtained from OpenStreetMap and analysed with `networkx`. This approach gives full control over the data and the analysis methods, but requires more computational resources and data preparation.

In the next section we will look at an alternative approach — using specialized routing services (such as OpenRouteService, OSRM, and others), which handle the calculations on their own side.